# 01 — Data Cleaning & Feature Engineering
> NovaCred Bank | Loan Default Risk Analysis

**Goal:** Load the raw Lending Club dataset, handle missing values, engineer features, and export a clean dataset for EDA and modelling.


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

## 1. Load Raw Data

In [ ]:
df = pd.read_csv('../data/raw/loan.csv', low_memory=False)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()[:10]} ...')
df.head(3)

## 2. Filter Target Variable

Keep only **Fully Paid** and **Charged Off** rows — these are the only clear-cut labels for binary classification.

In [ ]:
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
df['default'] = (df['loan_status'] == 'Charged Off').astype(int)

print(f'Rows after filter: {len(df):,}')
print(df['default'].value_counts(normalize=True).round(3))

## 3. Drop High-Missingness Columns

Columns with >40% missing values add noise without signal.

In [ ]:
missing_pct = df.isnull().sum() / len(df) * 100
cols_to_drop = missing_pct[missing_pct > 40].index.tolist()
print(f'Dropping {len(cols_to_drop)} columns with >40% missing values')
df.drop(columns=cols_to_drop, inplace=True)
print(f'Remaining columns: {df.shape[1]}')

## 4. Handle Remaining Nulls

In [ ]:
# Numeric: fill with median (robust to outliers)
num_cols = df.select_dtypes(include='number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Categorical: fill with mode
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print(f'Remaining nulls: {df.isnull().sum().sum()}')

## 5. Feature Engineering

In [ ]:
# Normalise column names if using smaller dataset
df.rename(columns={
    'annualInc': 'annual_inc',
    'homeOwnership': 'home_ownership',
    'empLength': 'emp_length'
}, inplace=True)

# Clean interest rate
if df['int_rate'].dtype == object:
    df['int_rate'] = df['int_rate'].str.replace('%', '').astype(float)

# Derived features
df['loan_to_income'] = df['loan_amnt'] / (df['annual_inc'] + 1)

# Parse issue date
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y', errors='coerce')
df['issue_year']  = df['issue_d'].dt.year
df['issue_month'] = df['issue_d'].dt.month

# Ordinal encode credit grade
grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
df['grade_num'] = df['grade'].map(grade_map)

print('Feature engineering complete ✅')
print(df[['loan_amnt','int_rate','loan_to_income','grade_num','default']].describe().round(3))

## 6. Export Cleaned Data

In [ ]:
import os
os.makedirs('../data/cleaned', exist_ok=True)
df.to_csv('../data/cleaned/loan_cleaned.csv', index=False)
print(f'Saved → data/cleaned/loan_cleaned.csv  ({df.shape[0]:,} rows, {df.shape[1]} cols)')